# [PlotPot](https://github.com/cryotud/plotpot) SEC-MALS plotting notebook 

Creates publication-quality SEC-MALS chromatograms from ASTRA exports.

**Input format:** Tab-separated ASTRA export (`.txt`). Number format is auto-detected — both European (comma as decimal) and standard (period as decimal) are supported.

**Expected column layout (repeating per run):**
```
Columns 0–7 per run:  (vol, UV), (vol, dRI), (vol, UV2), (vol, Mw_Da)
```
Runs are listed in file order. BSA is auto-detected by name. Mw column contains data only inside the integration window set in ASTRA.

**Workflow** [run cells top to bottom]:
1. **Dependencies** & **Imports** — run once.
2. **Upload**: file picker appears; select your ASTRA `.txt` export.
3. **Labels & options**: select runs to plot, set volume windows; re-run to update.
4. **Overview**: all runs, UV and dRI+Mw sanity check.
5. **Publication plot**: selected runs + BSA, will plot publication-quality panels.
6. **Save & download**: writes PDF + PNG and downloads to your machine.

In [ ]:
#@title Step 1 · Install & verify dependencies { display-mode: "form" }
import importlib, subprocess, sys

_required = ['numpy', 'pandas', 'matplotlib']
_missing  = [p for p in _required if importlib.util.find_spec(p) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + _missing)
    print(f'Installed: {_missing}')
else:
    print('All dependencies present:', _required)

In [ ]:
#@title Step 2 · Imports & plot style { display-mode: "form" }
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 10,
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.direction': 'out',
    'ytick.direction': 'out',
    'pdf.fonttype': 42,
    'svg.fonttype': 'none',
})

In [ ]:
#@title Step 3 · Upload data { display-mode: "form" }
from google.colab import files as _colab_files

print('A file picker will appear below — select your ASTRA export:')
_uploaded = _colab_files.upload()

_fname    = next(iter(_uploaded))
DATA_FILE = Path(_fname)
_content  = _uploaded[_fname].decode('utf-8')

# ── Format detection: 12 columns → conjugate, 8n columns → standard ───────────
_header_ncols = len(_content.split('\n')[0].rstrip('\r').split('\t'))
_AUTO_TYPE    = 'conjugate' if _header_ncols == 12 else 'standard'
print(f'Format detected: {_AUTO_TYPE!r}  ({_header_ncols} columns in header)\n')


# ── Helpers ───────────────────────────────────────────────────────────────────
def _detect_number_format(sample_parts):
    for p in sample_parts:
        p = p.strip()
        if not p or p in ('nan', 'NaN', 'None'):
            continue
        if p.startswith(',') or p.startswith('-,'):
            return 'european'
        if ',' in p and '.' in p:
            return 'european' if p.rindex(',') > p.rindex('.') else 'standard'
    return 'standard'


def _make_number_parser(fmt):
    if fmt == 'european':
        def _parse(s):
            s = str(s).strip()
            if s in ('', 'nan', 'NaN', 'None'):
                return np.nan
            s = s.replace('.', '').replace(',', '.')
            try:
                return float(s)
            except ValueError:
                return np.nan
    else:
        def _parse(s):
            s = str(s).strip()
            if s in ('', 'nan', 'NaN', 'None'):
                return np.nan
            s = s.replace(',', '')
            try:
                return float(s)
            except ValueError:
                return np.nan
    return _parse


def load_astra_multi(src):
    """Parse a standard ASTRA multi-run export (.txt)."""
    PAIRS = ['uv', 'ri', 'uv2', 'mw']
    if hasattr(src, 'read'):
        fh, _close = src, False
    else:
        fh, _close = open(src, encoding='utf-8'), True
    runs_order, col_map, pair_count = [], {}, {}
    raw_rows = []
    number_fmt = 'standard'
    parse = None
    try:
        for lineno, line in enumerate(fh):
            parts = line.rstrip('\n').split('\t')
            if lineno == 0:
                for i in range(0, len(parts) - 1, 2):
                    label = parts[i + 1].split('[')[0].strip()
                    if label not in col_map:
                        col_map[label] = {}
                        runs_order.append(label)
                        pair_count[label] = 0
                    idx = pair_count[label]
                    if idx < len(PAIRS):
                        col_map[label][PAIRS[idx]] = i
                    pair_count[label] += 1
                continue
            if parse is None:
                number_fmt = _detect_number_format(parts)
                parse = _make_number_parser(number_fmt)
            raw_rows.append([parse(p) for p in parts])
    finally:
        if _close:
            fh.close()
    raw = pd.DataFrame(raw_rows)
    runs_data = {}
    for label in runs_order:
        d = {}
        for pair, vcol in col_map[label].items():
            scol = vcol + 1
            sig  = 'mw_da' if pair == 'mw' else pair
            d[f'{pair}_vol'] = raw.iloc[:, vcol].values if vcol < raw.shape[1] else np.full(len(raw), np.nan)
            d[sig]           = raw.iloc[:, scol].values if scol < raw.shape[1] else np.full(len(raw), np.nan)
        runs_data[label] = pd.DataFrame(d)
    return runs_order, runs_data, number_fmt


def load_conjugate_export(src):
    """
    12-column ASTRA protein-conjugate export (English number format).
    Cols 0-1: UV, 2-3: dRI, 4-5: second detector,
    6-7: total Mw (Da), 8-9: protein Mw (Da), 10-11: component 2 Mw (Da).
    The three Mw pairs are non-empty only inside the ASTRA integration window.
    """
    col_names = [
        'uv_vol', 'uv', 'ri_vol', 'ri', 'sig2_vol', 'sig2',
        'mw_total_vol', 'mw_total_da',
        'mw_prot_vol',  'mw_prot_da',
        'mw_comp2_vol', 'mw_comp2_da',
    ]
    rows = []
    if hasattr(src, 'read'):
        lines = src.read().splitlines()
    else:
        with open(src, encoding='utf-8') as f:
            lines = f.read().splitlines()
    for i, line in enumerate(lines):
        if i == 0:
            continue
        parts = line.split('\t')
        vals = []
        for p in parts[:12]:
            p = p.strip()
            try:
                vals.append(float(p))
            except (ValueError, TypeError):
                vals.append(np.nan)
        while len(vals) < 12:
            vals.append(np.nan)
        rows.append(vals)
    return pd.DataFrame(rows, columns=col_names)


# ── Load ──────────────────────────────────────────────────────────────────────
df_conj     = None
runs_order  = []
runs_data   = {}
_fmt        = 'standard'
bsa_run     = ''
sample_runs = []

if _AUTO_TYPE == 'conjugate':
    df_conj = load_conjugate_export(io.StringIO(_content))
    print(f'Loaded {len(df_conj):,} rows from {_fname!r}\n')
    for _vc, _mc, _lb in [
        ('mw_total_vol', 'mw_total_da', 'Total'),
        ('mw_prot_vol',  'mw_prot_da',  'Component 1 (protein)'),
        ('mw_comp2_vol', 'mw_comp2_da', 'Component 2'),
    ]:
        _sub = df_conj.dropna(subset=[_vc, _mc])
        if len(_sub):
            print(f'  {_lb:<24}: {_sub[_mc].median()/1e3:.1f} kDa  '
                  f'(range {_sub[_mc].min()/1e3:.1f}–{_sub[_mc].max()/1e3:.1f} kDa, n={len(_sub)})')
        else:
            print(f'  {_lb:<24}: no data')
    print('\nProceed to Step 4 to set labels and volume window.')

else:
    runs_order, runs_data, _fmt = load_astra_multi(io.StringIO(_content))
    bsa_run     = next((r for r in runs_order if 'bsa' in r.lower()), runs_order[0] if runs_order else '')
    sample_runs = [r for r in runs_order if r != bsa_run]
    print(f'Loaded {_fname!r}  ({len(runs_data[runs_order[0]]) if runs_order else 0:,} rows)')
    print(f'Number format detected: {_fmt}\n')
    print(f'{len(runs_order)} runs found  (BSA auto-detected: {bsa_run!r})\n')
    for label in runs_order:
        tag = ' <- BSA' if label == bsa_run else ''
        mw  = runs_data[label]['mw_da'].dropna()
        mws = f'  Mw: {mw.min()/1e3:.1f}-{mw.max()/1e3:.1f} kDa (n={len(mw)})' if len(mw) else '  Mw: none'
        print(f'  * {label!r}{tag}{mws}')
    print()
    print('Paste these labels into SELECTED_RUNS in Step 4:')
    print('  ' + ', '.join(sample_runs))

In [ ]:
#@title Step 4 · Labels & options { display-mode: "form" }
#@markdown Edit fields below, then **Run** (Shift+Enter).

#@markdown **Data type** — auto-detected in Step 3; override only if needed.
DATA_TYPE = "auto" #@param ["auto", "standard", "conjugate"]
_DATA_TYPE = _AUTO_TYPE if DATA_TYPE == "auto" else DATA_TYPE
print(f'Data type: {_DATA_TYPE}  (auto-detected: {_AUTO_TYPE})\n')

#@markdown ---
#@markdown ### Standard options *(active when data type = standard)*
#@markdown Copy run labels from the Step 3 output.
SELECTED_RUNS    = ""    #@param {type:"string"}
RUN_LABELS_STR   = ""    #@param {type:"string"}
BSA_RUN_OVERRIDE = ""    #@param {type:"string"}
BSA_LABEL        = "BSA" #@param {type:"string"}
SHOW_BSA         = True  #@param {type:"boolean"}
#@markdown Sample volume window (mL) — one value or comma-separated per run:
SAMPLE_VOL_MIN_STR = "8.0"  #@param {type:"string"}
SAMPLE_VOL_MAX_STR = "20.0" #@param {type:"string"}
#@markdown BSA volume window (mL):
BSA_VOL_MIN = 13.0 #@param {type:"number"}
BSA_VOL_MAX = 17.0 #@param {type:"number"}

#@markdown ---
#@markdown ### Conjugate options *(active when data type = conjugate)*
SAMPLE_LABEL_CONJ  = "Sample"  #@param {type:"string"}
PROTEIN_LABEL_CONJ = "Protein" #@param {type:"string"}
COMP2_LABEL_CONJ   = "DNA"     #@param {type:"string"}
#@markdown Volume window (mL):
CONJ_VOL_MIN = 8.0  #@param {type:"number"}
CONJ_VOL_MAX = 13.0 #@param {type:"number"}

#@markdown ---
#@markdown ### Shared options
MW_YLIM_MIN = 1     #@param {type:"number"}
MW_YLIM_MAX = 10000 #@param {type:"number"}
#@markdown Mw filter — hide dots where dRI < this fraction of peak height:
MW_RI_THRESHOLD = 0.10 #@param {type:"slider", min:0.0, max:0.5, step:0.01}

MW_YLIM   = (MW_YLIM_MIN, MW_YLIM_MAX)
_CONJ_VOL = (CONJ_VOL_MIN, CONJ_VOL_MAX)

if _DATA_TYPE == 'standard':
    _active_bsa = BSA_RUN_OVERRIDE.strip() if BSA_RUN_OVERRIDE.strip() else bsa_run
    if SELECTED_RUNS.strip():
        _sel     = [s.strip() for s in SELECTED_RUNS.split(',') if s.strip() in runs_data]
        _missing = [s.strip() for s in SELECTED_RUNS.split(',') if s.strip() not in runs_data]
        if _missing:
            print(f'WARNING: runs not found (check spelling): {_missing}')
    else:
        _sel = list(sample_runs)

    def _parse_vol_str(s, n):
        vals = [float(x.strip()) for x in str(s).split(',') if x.strip()]
        if len(vals) == 1:
            return vals * n
        return (vals + [vals[-1]] * n)[:n]

    n_sel         = len(_sel)
    VOL_MINS      = _parse_vol_str(SAMPLE_VOL_MIN_STR, n_sel)
    VOL_MAXS      = _parse_vol_str(SAMPLE_VOL_MAX_STR, n_sel)
    BSA_VOL_RANGE = (BSA_VOL_MIN, BSA_VOL_MAX)
    _custom       = [l.strip() for l in RUN_LABELS_STR.split(',')]
    _labels       = [_custom[i] if i < len(_custom) and _custom[i] else r for i, r in enumerate(_sel)]

    print(f'BSA run  : {_active_bsa!r}  (show = {SHOW_BSA})')
    print(f'Selected runs ({n_sel}):')
    for r, lbl, vmin, vmax in zip(_sel, _labels, VOL_MINS, VOL_MAXS):
        mw  = runs_data[r]['mw_da'].dropna() if r in runs_data else pd.Series(dtype=float)
        mws = f'median {mw.median()/1e3:.0f} kDa' if len(mw) else 'no Mw'
        tag = f' (label: {lbl!r})' if lbl != r else ''
        print(f'  • {r!r}{tag}  vol {vmin}–{vmax} mL  ({mws})')
    if SHOW_BSA:
        print(f'BSA panel: vol {BSA_VOL_MIN}–{BSA_VOL_MAX} mL')
    print(f'Mw axis  : {MW_YLIM_MIN}–{MW_YLIM_MAX} kDa (log)')

else:
    print(f'Sample      : {SAMPLE_LABEL_CONJ!r}')
    print(f'Component 1 : {PROTEIN_LABEL_CONJ!r}')
    print(f'Component 2 : {COMP2_LABEL_CONJ!r}')
    print(f'Volume      : {CONJ_VOL_MIN}–{CONJ_VOL_MAX} mL')
    print(f'Mw axis     : {MW_YLIM_MIN}–{MW_YLIM_MAX} kDa (log)')

In [ ]:
#@title Step 5 · Overview { display-mode: "form" }

if _DATA_TYPE == 'standard':
    n_runs = len(runs_order)
    fig_ov, axes_ov = plt.subplots(n_runs, 2, figsize=(12, 3.5 * n_runs), squeeze=False, layout='constrained')
    for row, label in enumerate(runs_order):
        df_r   = runs_data[label]
        ax_uv  = axes_ov[row, 0]
        ax_dri = axes_ov[row, 1]
        uv = df_r[['uv_vol', 'uv']].dropna()
        ax_uv.plot(uv['uv_vol'], uv['uv'], color='steelblue', lw=0.8)
        ax_uv.set_title(f'{label} — UV', fontsize=10)
        ax_uv.set_xlabel('Volume (mL)')
        ax_uv.set_ylabel('UV (AU)')
        ax_uv.spines['top'].set_visible(False)
        ax_uv.spines['right'].set_visible(False)
        ri = df_r[['ri_vol', 'ri']].dropna()
        ax_dri.plot(ri['ri_vol'], ri['ri'], color='forestgreen', lw=0.8)
        ax_dri.set_title(f'{label} — dRI + Mw', fontsize=10)
        ax_dri.set_xlabel('Volume (mL)')
        ax_dri.set_ylabel('dRI')
        ax_dri.spines['top'].set_visible(False)
        ax_mw2 = ax_dri.twinx()
        mw = df_r[['mw_vol', 'mw_da']].dropna()
        if len(mw):
            ax_mw2.scatter(mw['mw_vol'], mw['mw_da'] / 1e3, color='firebrick', s=3, zorder=5, alpha=0.8)
        ax_mw2.set_yscale('log')
        ax_mw2.set_ylim(*MW_YLIM)
        ax_mw2.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:g}'))
        ax_mw2.set_ylabel('Molar mass (kDa)', color='firebrick')
        ax_mw2.tick_params(axis='y', labelcolor='firebrick')
    fig_ov.suptitle(f'{DATA_FILE.stem} — overview ({n_runs} runs)', fontsize=13)

else:  # conjugate
    fig_ov, (ax_uv_ov, ax_ri_ov) = plt.subplots(1, 2, figsize=(12, 3.5), layout='constrained')
    uv = df_conj[['uv_vol', 'uv']].dropna()
    ax_uv_ov.plot(uv['uv_vol'], uv['uv'], color='steelblue', lw=0.8)
    ax_uv_ov.set_title('UV', fontsize=10)
    ax_uv_ov.set_xlabel('Volume (mL)')
    ax_uv_ov.set_ylabel('UV (AU)')
    ax_uv_ov.spines['top'].set_visible(False)
    ax_uv_ov.spines['right'].set_visible(False)
    ri = df_conj[['ri_vol', 'ri']].dropna()
    ax_ri_ov.plot(ri['ri_vol'], ri['ri'], color='forestgreen', lw=0.8)
    ax_ri_ov.set_title('dRI + Mw components', fontsize=10)
    ax_ri_ov.set_xlabel('Volume (mL)')
    ax_ri_ov.set_ylabel('dRI')
    ax_ri_ov.spines['top'].set_visible(False)
    ax_mw_ov = ax_ri_ov.twinx()
    for _vc, _mc, _col, _lb in [
        ('mw_total_vol', 'mw_total_da', '#e8b004', 'Total'),
        ('mw_prot_vol',  'mw_prot_da',  '#1a4f8a', PROTEIN_LABEL_CONJ),
        ('mw_comp2_vol', 'mw_comp2_da', '#c0392b', COMP2_LABEL_CONJ),
    ]:
        _sub = df_conj[[_vc, _mc]].dropna()
        if len(_sub):
            ax_mw_ov.scatter(_sub[_vc], _sub[_mc] / 1e3, color=_col, s=3, zorder=5, alpha=0.8, label=_lb)
    ax_mw_ov.set_yscale('log')
    ax_mw_ov.set_ylim(*MW_YLIM)
    ax_mw_ov.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:g}'))
    ax_mw_ov.set_ylabel('Molar mass (kDa)')
    ax_mw_ov.legend(fontsize=8, frameon=False)
    fig_ov.suptitle(f'{DATA_FILE.stem} — overview (conjugate)', fontsize=13)

plt.show()

In [ ]:
#@title Step 6 · Publication plot { display-mode: "form" }
#@markdown *Adjust selection and windows in Step 4, then run this cell.*

from matplotlib.transforms import blended_transform_factory as _btf

if _DATA_TYPE == 'standard':
    SAMPLE_COLORS = ['#1a4f8a', '#c0392b', '#27ae60', '#8e44ad', '#d35400']
    C_BSA = '#888888'
    C_MW  = '#f0c106'

    show_bsa_panel = SHOW_BSA and bool(_active_bsa) and _active_bsa in runs_data
    n_panels       = len(_sel) + (1 if show_bsa_panel else 0)

    if n_panels == 0:
        print('No runs to plot — check SELECTED_RUNS in Step 4.')
    else:
        fig, axes_p = plt.subplots(1, n_panels, figsize=(4.5 * n_panels, 3.8),
                                    layout='constrained')
        if n_panels == 1:
            axes_p = [axes_p]

        _fig_h_pt    = fig.get_figheight() * 72
        _log_y_range = np.log10(MW_YLIM[1]) - np.log10(MW_YLIM[0])
        _axes_h_pt   = _fig_h_pt * 0.55
        _label_h_pt  = 14.0
        _MIN_LOG_GAP = max(_label_h_pt / (_axes_h_pt / _log_y_range), 0.08)

        def _deoverlap_log(idxs, label_ys):
            if len(idxs) < 2:
                return
            true_log = np.array([np.log10(label_ys[i]) for i in idxs])
            result   = true_log.copy()
            for k in range(1, len(result)):
                if result[k] < result[k - 1] + _MIN_LOG_GAP:
                    result[k] = result[k - 1] + _MIN_LOG_GAP
            for k in range(len(result) - 2, -1, -1):
                if result[k] > result[k + 1] - _MIN_LOG_GAP:
                    result[k] = result[k + 1] - _MIN_LOG_GAP
            result += np.mean(true_log) - np.mean(result)
            for i, y in zip(idxs, 10.0 ** result):
                label_ys[i] = y

        def _pub_panel(ax_mw, df_r, vmin, vmax, label, color):
            ax_dri = ax_mw.twinx()
            ri = df_r[['ri_vol', 'ri']].dropna()
            m  = (ri['ri_vol'] >= vmin) & (ri['ri_vol'] <= vmax)
            v, s = ri.loc[m, 'ri_vol'].values, ri.loc[m, 'ri'].values
            s_norm = s / s.max() if len(s) and s.max() > 0 else s
            ax_dri.plot(v, s_norm, color=color, lw=1.5, label=label)
            ax_dri.set_ylim(-0.06, 1.05)
            ax_dri.set_ylabel('Normalized dRI', fontsize=11)

            mw = df_r[['mw_vol', 'mw_da']].dropna()
            mw = mw[(mw['mw_vol'] >= vmin) & (mw['mw_vol'] <= vmax)].copy()
            if len(mw) > 0 and len(v) > 0:
                ri_at = np.interp(mw['mw_vol'].values, v, s_norm)
                mw    = mw[ri_at >= MW_RI_THRESHOLD]

            mw_kda = mw['mw_da'].values / 1e3
            vol_mw = mw['mw_vol'].values
            ax_mw.scatter(vol_mw, mw_kda, color=C_MW, s=4, zorder=5, alpha=0.8, label='Molar mass')

            if len(mw_kda):
                order    = np.argsort(vol_mw)
                vw, mk   = vol_mw[order], mw_kda[order]
                dv       = np.diff(vw)
                med_step = float(np.median(dv[dv > 0])) if np.any(dv > 0) else 1.0
                splits   = np.where(dv > 10 * med_step)[0] + 1
                segs     = []
                for vg, mg in zip(np.split(vw, splits), np.split(mk, splits)):
                    if not len(mg):
                        continue
                    med   = float(np.median(mg))
                    ax_mw.axhline(med, color=C_MW, lw=0.8, ls='--', alpha=0.5)
                    x_rel = (float(np.mean(vg)) - vmin) / (vmax - vmin) if vmax > vmin else 0.5
                    x_ax, ha = (0.97, 'right') if x_rel < 0.5 else (0.03, 'left')
                    segs.append((x_ax, ha, med))
                label_ys = [s[2] for s in segs]
                for side in (0.97, 0.03):
                    idxs = sorted([i for i in range(len(segs)) if segs[i][0] == side],
                                   key=lambda i: segs[i][2])
                    _deoverlap_log(idxs, label_ys)
                for (x_ax, ha, true_med), label_y in zip(segs, label_ys):
                    ax_mw.text(x_ax, label_y, f'{true_med:.0f} kDa',
                               transform=_btf(ax_mw.transAxes, ax_mw.transData),
                               color=C_MW, fontsize=9, va='center', ha=ha,
                               bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.85))

            ax_mw.set_xlim(vmin, vmax)
            ax_mw.set_ylim(*MW_YLIM)
            ax_mw.set_yscale('log')
            ax_mw.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:g}'))
            ax_mw.set_xlabel('Elution volume (mL)', fontsize=11)
            ax_mw.set_ylabel('Molar mass (kDa)', fontsize=11)
            ax_mw.set_title(label, fontsize=11, fontweight='bold')
            for spine in ax_mw.spines.values():
                spine.set_visible(True)
            for spine in ax_dri.spines.values():
                spine.set_visible(True)
            h1, l1 = ax_dri.get_legend_handles_labels()
            h2, l2 = ax_mw.get_legend_handles_labels()
            ax_mw.legend(h1 + h2, l1 + l2, loc='upper right', frameon=True,
                         facecolor='white', edgecolor='none', framealpha=0.85, fontsize=9)

        for i, (run_label, disp_label, vmin, vmax) in enumerate(zip(_sel, _labels, VOL_MINS, VOL_MAXS)):
            _pub_panel(axes_p[i], runs_data[run_label], vmin, vmax,
                       disp_label, SAMPLE_COLORS[i % len(SAMPLE_COLORS)])
        if show_bsa_panel:
            _pub_panel(axes_p[-1], runs_data[_active_bsa], BSA_VOL_MIN, BSA_VOL_MAX,
                       BSA_LABEL, C_BSA)
        plt.show()

else:  # conjugate
    _C_TOTAL   = '#e8b004'
    _C_PROTEIN = '#1a4f8a'
    _C_COMP2   = '#c0392b'
    _C_SIG     = '#555555'

    def _slice_c(df, vcol, scol, vmin, vmax):
        m = df[[vcol, scol]].notna().all(axis=1) & (df[vcol] >= vmin) & (df[vcol] <= vmax)
        return df.loc[m, vcol].values, df.loc[m, scol].values

    fig, ax_mw_c = plt.subplots(1, 1, figsize=(5, 4))
    ax_sig_c = ax_mw_c.twinx()

    _vs, _ss = _slice_c(df_conj, 'ri_vol', 'ri', *_CONJ_VOL)
    _ss_norm = _ss / _ss.max() if len(_ss) and _ss.max() > 0 else _ss
    ax_sig_c.plot(_vs, _ss_norm, color=_C_SIG, lw=1.5, label=SAMPLE_LABEL_CONJ)
    ax_sig_c.set_ylim(-0.06, 1.05)
    ax_sig_c.set_ylabel('Normalized dRI', fontsize=11)

    _mw_series_c = [
        ('mw_total_vol', 'mw_total_da', _C_TOTAL,   'Total'),
        ('mw_prot_vol',  'mw_prot_da',  _C_PROTEIN, PROTEIN_LABEL_CONJ),
        ('mw_comp2_vol', 'mw_comp2_da', _C_COMP2,   COMP2_LABEL_CONJ),
    ]
    _entries_c = []
    for _vcol, _mcol, _color, _lbl in _mw_series_c:
        _v, _m = _slice_c(df_conj, _vcol, _mcol, *_CONJ_VOL)
        if not len(_v):
            continue
        _ri_at = np.interp(_v, _vs, _ss_norm, left=0.0, right=0.0) if len(_vs) else np.zeros(len(_v))
        _mask  = _ri_at >= MW_RI_THRESHOLD
        _vf, _kda_f = _v[_mask], _m[_mask] / 1e3
        ax_mw_c.scatter(_vf, _kda_f, color=_color, s=4, zorder=5, alpha=0.9, label=_lbl)
        if len(_kda_f):
            _med = float(np.median(_kda_f))
            ax_mw_c.axhline(_med, color=_color, lw=0.8, ls='--', alpha=0.5)
            _entries_c.append((_med, _color, _lbl))

    _entries_c.sort(key=lambda e: e[0])
    _label_ys_c  = [e[0] for e in _entries_c]
    _min_log_gap = 0.15
    _changed     = True
    while _changed:
        _changed = False
        for _k in range(1, len(_label_ys_c)):
            if np.log10(_label_ys_c[_k]) - np.log10(_label_ys_c[_k - 1]) < _min_log_gap:
                _mid = 0.5 * (np.log10(_label_ys_c[_k]) + np.log10(_label_ys_c[_k - 1]))
                _label_ys_c[_k - 1] = 10 ** (_mid - _min_log_gap / 2)
                _label_ys_c[_k]     = 10 ** (_mid + _min_log_gap / 2)
                _changed = True

    _trans_c = _btf(ax_mw_c.transAxes, ax_mw_c.transData)
    for (_med, _color, _lbl), _ly in zip(_entries_c, _label_ys_c):
        ax_mw_c.text(1.02, _ly, f'{_med:.0f} kDa\n({_lbl})',
                     transform=_trans_c, color=_color, fontsize=8,
                     va='center', ha='left', clip_on=False)
        if abs(np.log10(_ly / _med)) > 0.02:
            ax_mw_c.plot([_CONJ_VOL[1], _CONJ_VOL[1]], [_med, _ly],
                         color=_color, lw=0.4, ls=':', clip_on=False, zorder=4)

    ax_mw_c.set_xlim(_CONJ_VOL)
    ax_mw_c.set_ylim(*MW_YLIM)
    ax_mw_c.set_yscale('log')
    ax_mw_c.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:g}'))
    ax_mw_c.set_xlabel('Elution volume (mL)', fontsize=11)
    ax_mw_c.set_ylabel('Molar mass (kDa)', fontsize=11)
    ax_mw_c.set_title(SAMPLE_LABEL_CONJ, fontsize=11, fontweight='bold')
    for _ax in (ax_mw_c, ax_sig_c):
        for _spine in _ax.spines.values():
            _spine.set_visible(True)
    _h1, _l1 = ax_sig_c.get_legend_handles_labels()
    _h2, _l2 = ax_mw_c.get_legend_handles_labels()
    ax_mw_c.legend(_h1 + _h2, _l1 + _l2, loc='upper right', frameon=True,
                   facecolor='white', edgecolor='none', framealpha=0.85, fontsize=8)
    fig.subplots_adjust(right=0.70)
    plt.show()

In [ ]:
#@title Step 7 · Save & download { display-mode: "form" }
from google.colab import files as _colab_files

OUTPUT_STEM = DATA_FILE.stem
_suffix     = 'conjugate' if _DATA_TYPE == 'conjugate' else 'sec_mals'

_outputs = []
for ext in ('pdf', 'png'):
    out = f'/content/{OUTPUT_STEM}_{_suffix}.{ext}'
    fig.savefig(out, dpi=300, bbox_inches='tight')
    print(f'Saved → {out}')
    _outputs.append(out)

print('\nStarting downloads...')
for out in _outputs:
    _colab_files.download(out)